# Two-Agent Equilibrium with State-Contingent Assets

Each agent has endowment
$$
e^i=(e_0^i,e_g^i,e_b^i)
$$
and preferences
$$
U^i=c_0^i+\beta\left[\pi_g u(x_g^i)+\pi_b u(x_b^i)\right],
\qquad \pi_b=1-\pi_g,
$$
where
$$
u(x)=
\begin{cases}
\dfrac{x^{1-\rho}-1}{1-\rho}, & \rho\neq 1,\\[6pt]
\ln x, & \rho=1,
\end{cases}
\qquad \rho\ge 0.
$$

Thus
$$
u'(x)=x^{-\rho}.
$$

Asset $j$ has payoff $r^j=(r_g^j,r_b^j)$, date-0 price $q_j$, and zero net supply.

If agent 1 holds $a$, agent 2 holds $-a$.


## 1. Inputs

Change only this cell.

In [1]:
import numpy as np
from scipy.optimize import least_squares

endowment_1 = np.array([1.0, 2.0, 0.0])
endowment_2 = np.array([1.0, 0.0, 2.0])

beta = 0.95
rho = 1.0       # rho >= 0; rho = 1 gives log utility
pi_g = 0.5
pi_b = 1 - pi_g

# Each row is [payoff in Good, payoff in Bad]
assets = np.array([
    [1.0, -1.0]
], dtype=float)

# No assets:
# assets = np.empty((0,2))

# Two Arrow securities:
# assets = np.array([[1.0,0.0],[0.0,1.0]])

## 2. Equilibrium conditions

Since date-0 utility is linear, its marginal utility is 1. For asset $j$, agent $i$'s FOC is
$$
q_j=\beta\left[
\pi_g r_g^j(x_g^i)^{-\rho}
+\pi_b r_b^j(x_b^i)^{-\rho}
\right].
$$

The solver chooses asset prices $q$ and agent 1's portfolio $a$, with agent 2 holding $-a$, so that both agents' FOCs hold simultaneously.

The numerical routine searches for an interior equilibrium.


In [2]:
def prepare_assets(assets):
    A = np.asarray(assets, dtype=float)
    if A.size == 0:
        return np.empty((0,2))
    if A.ndim == 1:
        A = A.reshape(1,2)
    if A.shape[1] != 2:
        raise ValueError("Each asset must be [r_g, r_b].")
    return A

def get_consumption(a, q, e1, e2, A):
    a2 = -a
    c01 = e1[0] - q @ a
    c02 = e2[0] - q @ a2

    if len(a) == 0:
        return e1.copy(), e2.copy()

    c1 = np.array([
        c01,
        e1[1] + A[:,0] @ a,
        e1[2] + A[:,1] @ a
    ])
    c2 = np.array([
        c02,
        e2[1] + A[:,0] @ a2,
        e2[2] + A[:,1] @ a2
    ])
    return c1, c2

def residuals(z, e1, e2, beta, rho, pi_g, A):
    J = len(A)
    a = z[:J]
    q = z[J:]
    c1, c2 = get_consumption(a, q, e1, e2, A)
    pg, pb = pi_g, 1-pi_g

    if min(c1[0],c1[1],c1[2],c2[0],c2[1],c2[2]) <= 0:
        return np.ones(2*J)*1e6

    val1 = beta*(pg*A[:,0]*c1[1]**(-rho) + pb*A[:,1]*c1[2]**(-rho))
    val2 = beta*(pg*A[:,0]*c2[1]**(-rho) + pb*A[:,1]*c2[2]**(-rho))

    return np.r_[q-val1, q-val2]

def solve_equilibrium(e1, e2, beta, rho, pi_g, assets, guess=None):
    e1 = np.asarray(e1,float)
    e2 = np.asarray(e2,float)
    A = prepare_assets(assets)
    J = len(A)

    if rho < 0:
        raise ValueError("rho must be nonnegative.")

    if J == 0:
        return {
            "success": True,
            "prices": np.array([]),
            "portfolio_1": np.array([]),
            "portfolio_2": np.array([]),
            "consumption_1": e1.copy(),
            "consumption_2": e2.copy(),
            "rank": 0,
            "max_residual": 0.0
        }

    if guess is None:
        # Search for a feasible initial portfolio, then use zero initial prices.
        rng = np.random.default_rng(1234)
        a0 = np.zeros(J)
        found = False

        for scale in [0.05,0.1,0.2,0.4,0.8]:
            for _ in range(5000):
                trial = rng.normal(size=J)*scale
                c1,c2 = get_consumption(trial,np.zeros(J),e1,e2,A)
                if min(c1[0],c1[1],c1[2],c2[0],c2[1],c2[2]) > 1e-5:
                    a0 = trial
                    found = True
                    break
            if found:
                break

        guess = np.r_[a0, np.zeros(J)]

    sol = least_squares(
        residuals, guess,
        args=(e1,e2,beta,rho,pi_g,A),
        xtol=1e-12, ftol=1e-12, gtol=1e-12,
        max_nfev=50000
    )

    a = sol.x[:J]
    q = sol.x[J:]
    c1,c2 = get_consumption(a,q,e1,e2,A)
    r = residuals(sol.x,e1,e2,beta,rho,pi_g,A)
    maxres = np.max(np.abs(r))

    feasible = min(c1[0],c1[1],c1[2],c2[0],c2[1],c2[2]) > 0

    return {
        "success": bool(sol.success and feasible and maxres < 1e-8),
        "prices": q,
        "portfolio_1": a,
        "portfolio_2": -a,
        "consumption_1": c1,
        "consumption_2": c2,
        "rank": np.linalg.matrix_rank(A),
        "max_residual": maxres,
        "solver": sol
    }

## 3. Solve

In [3]:
eq = solve_equilibrium(
    endowment_1,
    endowment_2,
    beta,
    rho,
    pi_g,
    assets
)

print("Equilibrium found:", eq["success"])
print("Asset payoff rank:", eq["rank"])
print("Asset prices:", eq["prices"])
print("Agent 1 portfolio:", eq["portfolio_1"])
print("Agent 2 portfolio:", eq["portfolio_2"])
print("Agent 1 consumption [c0, xg, xb]:", eq["consumption_1"])
print("Agent 2 consumption [c0, xg, xb]:", eq["consumption_2"])
print("Maximum residual:", eq["max_residual"])

Equilibrium found: True
Asset payoff rank: 1
Asset prices: [2.63032196e-21]
Agent 1 portfolio: [-1.]
Agent 2 portfolio: [1.]
Agent 1 consumption [c0, xg, xb]: [1. 1. 1.]
Agent 2 consumption [c0, xg, xb]: [1. 1. 1.]
Maximum residual: 6.686873540349514e-14


## 4. Utility

In [4]:
def u_crra(x, rho):
    if x <= 0:
        return -np.inf
    if np.isclose(rho, 1.0):
        return np.log(x)
    return (x**(1-rho) - 1)/(1-rho)

def utility(c, beta, rho, pi_g):
    pi_b = 1-pi_g
    if c[1] <= 0 or c[2] <= 0:
        return -np.inf
    return c[0] + beta*(
        pi_g*u_crra(c[1], rho)
        + pi_b*u_crra(c[2], rho)
    )

print("Agent 1 utility:", utility(eq["consumption_1"], beta, rho, pi_g))
print("Agent 2 utility:", utility(eq["consumption_2"], beta, rho, pi_g))


Agent 1 utility: 1.0
Agent 2 utility: 1.0


## 5. Market-clearing checks

Since every asset is in zero net supply,
$$
a^1+a^2=0.
$$

Aggregate consumption should also equal aggregate endowment in every commodity.

In [5]:
print("Asset-market clearing:")
print(eq["portfolio_1"] + eq["portfolio_2"])

print("\nGoods-market clearing:")
C = eq["consumption_1"] + eq["consumption_2"]
E = endowment_1 + endowment_2
print("Aggregate consumption:", C)
print("Aggregate endowment:  ", E)
print("Difference:           ", C-E)

Asset-market clearing:
[0.]

Goods-market clearing:
Aggregate consumption: [2. 2. 2.]
Aggregate endowment:   [2. 2. 2.]
Difference:            [0. 0. 0.]


## 6. Asset-pricing check

At equilibrium, both agents must value each asset at the same price:
$$
q_j
=
\beta\left[\pi_g r_g^j(x_g^1)^{-\rho}
+\pi_b r_b^j(x_b^1)^{-\rho}\right]
=
\beta\left[\pi_g r_g^j(x_g^2)^{-\rho}
+\pi_b r_b^j(x_b^2)^{-\rho}\right].
$$


In [6]:
if len(eq["prices"]) > 0 and eq["success"]:
    A = prepare_assets(assets)
    c1 = eq["consumption_1"]
    c2 = eq["consumption_2"]

    q1 = beta*(pi_g*A[:,0]*c1[1]**(-rho) + pi_b*A[:,1]*c1[2]**(-rho))
    q2 = beta*(pi_g*A[:,0]*c2[1]**(-rho) + pi_b*A[:,1]*c2[2]**(-rho))

    print("             q market       q from agent 1     q from agent 2")
    for j in range(len(eq["prices"])):
        print(f"Asset {j+1:>2}: {eq['prices'][j]:>12.6f} "
              f"{q1[j]:>18.6f} {q2[j]:>18.6f}")
else:
    print("No priced assets to check, or no equilibrium was found.")

             q market       q from agent 1     q from agent 2
Asset  1:     0.000000           0.000000          -0.000000


## 7. Completeness and redundancy

The future-state payoff matrix has two rows, one for each state. Its rank is therefore at most 2.

- rank 0: no effective asset span,
- rank 1: incomplete markets,
- rank 2: complete span of the two future states,
- more than two assets: some assets must be redundant.

With redundant assets, equilibrium consumption may be unique even though individual portfolio holdings are not.